<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/13%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_RAG_%E1%84%80%E1%85%A9%E1%84%83%E1%85%A9%E1%84%92%E1%85%AA_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 13주차 · RAG 고도화 실험 노트북

## 목표
> **12주차 RAG를 베이스로**, 고도화 기법을 하나씩 **실험**하며 효과를 눈으로 확인합니다.

이 노트북은 **실험 중심**입니다. 각 기법마다 **"적용 전 vs 적용 후"** 를 비교합니다.
파라미터(청크 크기·가중치·k 등)를 바꿔가며 자유롭게 돌려보세요. 🧪

### 🧪 실험 목록
| 파트 | 실험 | 셀 |
| --- | --- | --- |
| **A** | 12주차 베이스 재구성 (준용) | 준비 |
| **B** | 청킹 전략 | 1️⃣ 2️⃣ |
| **C** | 하이브리드 검색 (키워드+의미) | 3️⃣ 4️⃣ |
| **D** | MMR (다양성) | 5️⃣ |
| **E** | 리랭킹 (심화·선택) | 6️⃣ |
| **F** | 메타데이터 필터 | 7️⃣ |
| **G** | 프롬프트 고도화 | 8️⃣ |
| **H** | 출처 표시 | 9️⃣ |
| **I** | 종합 · Before vs After | 🔟 |

> 💡 **A(준비)를 먼저 실행**한 뒤 B~I를 원하는 순서로 실험하세요.
> 실험 도구 `compare_search`, `ask`, `make_store` 가 반복 실험을 쉽게 해줍니다.
> Google Colab 기준입니다.

---
## 0. 준비 — 설치 & API 키

13주차에서는 하이브리드 검색용 `rank_bm25`, `langchain-community`가 추가로 필요합니다.

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb langchain-text-splitters langchain-community rank_bm25

print("✅ 설치 완료!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Google API 키를 입력하세요: ")
print("✅ API 키 설정 완료!")

Google API 키를 입력하세요: ··········
✅ API 키 설정 완료!


---
# 📦 Part A. 12주차 베이스 재구성 (준용)

먼저 12주차 RAG를 그대로 세팅합니다. 앞으로 이 위에서 실험합니다.
실험이 잘 보이도록 문서를 **여러 섹션 + 식별자(정책번호·전화번호)** 로 조금 풍부하게 준비했습니다.

### A-1. 실험용 문서

In [ ]:
# 위니브마켓 고객 정책 (여러 섹션 + 식별자 포함)
policy_text = """위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)

[반품 신청 기간]
위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다.
반품 신청은 마이페이지 또는 고객센터(1588-0000)를 통해 접수할 수 있습니다.

[반품 배송비]
단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다.
다만 상품에 하자가 있거나 오배송된 경우에는 배송비 전액을 위니브마켓이 부담합니다.

[반품 불가 상품]
신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다.
또한 고객의 사용으로 상품 가치가 현저히 감소한 경우에도 반품이 제한될 수 있습니다.

[교환 안내]
교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다.
다른 상품으로의 교환은 반품 후 재주문으로 처리됩니다.

[환불 시점]
반품 상품이 물류센터에 도착해 검수가 완료되면,
영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행됩니다.

[배송 안내]
주문은 결제 완료 후 영업일 기준 1~2일 이내에 출고됩니다.
도서·산간 지역은 추가 배송비가 발생할 수 있으며, 배송 조회는 마이페이지에서 가능합니다.

[회원 등급 혜택]
위니브마켓 회원은 실버, 골드, VIP 등급으로 나뉩니다.
등급이 높을수록 적립률과 무료배송 혜택이 커집니다."""

print(f"📄 문서 길이: {len(policy_text)}자")

📄 문서 길이: 659자


### A-2. 임베딩 & 벡터 DB (+ 실험용 저장소 헬퍼)

`make_store()` 는 실험할 때마다 **깨끗한 벡터 DB**를 새로 만들어주는 함수입니다.
(셀을 다시 실행해도 데이터가 중복되지 않도록 매번 독립된 저장소를 만듭니다.)

In [ ]:
pip install -U langchain-google-genai

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 기존: model="models/embedding-001" (에러 발생)
# 변경: "text-embedding-004" 또는 "models/text-embedding-004"
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

def make_store(texts=None, documents=None, name="exp"):
    """독립된 인메모리 벡터 DB 생성 (재실행해도 안전)"""
    client = chromadb.EphemeralClient()
    if documents is not None:
        return Chroma.from_documents(documents=documents, embedding=embedding,
                                     client=client, collection_name=name)
    return Chroma.from_texts(texts=texts, embedding=embedding,
                             client=client, collection_name=name)

# 12주차 방식: 분할 → 저장 → 기본 리트리버
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
chunks = splitter.split_text(policy_text)

vectorstore = make_store(texts=chunks, name="base")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ 베이스 구축 완료 — 청크 {len(chunks)}개")

✅ 베이스 구축 완료 — 청크 9개


### A-3. 프롬프트 · LLM · 기본 RAG 체인 (12주차)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 기본 프롬프트 (12주차)
basic_prompt = ChatPromptTemplate.from_template(
    "아래 [컨텍스트]에만 근거해 답하세요. 없으면 '모른다'고 답하세요.\n\n"
    "[컨텍스트]\n{context}\n\n[질문]\n{question}"
)

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 기본 RAG 체인 (12주차)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | basic_prompt | llm | StrOutputParser()
)

print("✅ 기본 RAG 체인 준비 완료")

✅ 기본 RAG 체인 준비 완료


### A-4. 🧪 실험 도구 (앞으로 계속 사용)

- `show_docs(docs)` : 검색된 청크를 간략히 출력
- `compare_search(질문, {이름: 리트리버, ...})` : 여러 검색기 결과를 나란히 비교
- `ask(질문, chain)` : 체인에 질문하고 답변 출력 (기본값 = 기본 rag_chain)

In [ ]:
def show_docs(docs, n=65):
    """검색된 청크를 간략히 출력"""
    print(f"  📄 검색된 청크 {len(docs)}개")
    for i, d in enumerate(docs):
        meta = f"   {d.metadata}" if d.metadata else ""
        preview = d.page_content[:n].replace("\n", " ").strip()
        print(f"   [{i}] {preview} ...{meta}")

def compare_search(question, retrievers, n=55):
    """여러 리트리버의 검색 결과를 나란히 비교"""
    print(f"❓ 질문: {question}\n")
    for name, r in retrievers.items():
        print(f"── [{name}] " + "─" * 32)
        show_docs(r.invoke(question), n)
        print()

def ask(question, chain=None):
    """체인에 질문하고 답변 출력 (기본값: 기본 rag_chain)"""
    chain = chain or rag_chain
    print(f"❓ {question}")
    print(f"🤖 {chain.invoke(question)}\n")

print("✅ 실험 도구(show_docs / compare_search / ask) 준비 완료")

✅ 실험 도구(show_docs / compare_search / ask) 준비 완료


### A-5. 베이스 동작 확인

In [ ]:
ask("반품은 며칠 이내에 가능한가요?")

❓ 반품은 며칠 이내에 가능한가요?
🤖 상품 수령일로부터 14일 이내에 가능합니다.



---
# ✂️ Part B. 실험 — 청킹 전략  (1️⃣ 2️⃣)

**"검색의 8할은 잘 나누기."** `chunk_size`를 바꾸면 청크 개수와 검색 결과가 어떻게 달라질까요?

### 1️⃣ 실험 A — chunk_size별 청크 개수 비교

먼저 **저장 없이** 분할만 해서, 크기에 따라 몇 조각이 나오는지 봅니다.

In [ ]:
# 여러 chunk_size로 잘라보고 개수 비교
for size in [100, 300, 800]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    cks = sp.split_text(policy_text)
    print(f"chunk_size={size:>4} → 청크 {len(cks)}개  (평균 {len(policy_text)//len(cks)}자)")

chunk_size= 100 → 청크 9개  (평균 73자)
chunk_size= 300 → 청크 3개  (평균 219자)
chunk_size= 800 → 청크 1개  (평균 659자)


### 2️⃣ 실험 B — 청크 크기가 검색에 미치는 영향

`chunk_size=100`(잘게)으로 만든 저장소와 **베이스(300)** 의 검색 결과를 비교해 봅시다.
잘게 나누면 더 뾰족하게, 크게 나누면 더 넓게 잡히는 경향을 확인하세요.

> 🔧 실험: `size=100`을 다른 값으로 바꿔 보세요.

In [ ]:
# 잘게 나눈 저장소 만들기
size = 100
sp_small = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
chunks_small = sp_small.split_text(policy_text)
retriever_small = make_store(texts=chunks_small, name=f"chunk_{size}").as_retriever(search_kwargs={"k": 3})

# 베이스(300) vs 잘게(100) 검색 비교
compare_search(
    "반품 배송비는 누가 내나요?",
    {"베이스 (chunk 300)": retriever, f"잘게 (chunk {size})": retriever_small},
)

❓ 질문: 반품 배송비는 누가 내나요?

── [베이스 (chunk 300)] ────────────────────────────────
  📄 검색된 청크 3개
   [0] [반품 배송비] 단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다. 다만 ...
   [1] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)  [반품 신청 기간] 위니브마켓에서 ...
   [2] [교환 안내] 교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다. 다른 상품 ...

── [잘게 (chunk 100)] ────────────────────────────────
  📄 검색된 청크 3개
   [0] [반품 배송비] 단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다. 다만 ...
   [1] [반품 배송비] 단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다. 다만 ...
   [2] [교환 안내] 교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다. 다른 상품 ...



---
# 🔀 Part C. 실험 — 하이브리드 검색  (3️⃣ 4️⃣)

**키워드(BM25) + 의미(벡터)** 를 합칩니다. 특히 **고유명사·코드** 검색에서 차이가 큽니다.

### 3️⃣ 하이브리드 검색기 만들기 (EnsembleRetriever)

In [ ]:
!pip install -qU langchain langchain-community rank_bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever  # ← 변경된 경로

# 키워드 검색기 (BM25)
bm25 = BM25Retriever.from_texts(chunks)
bm25.k = 3

# 하이브리드 검색기 생성 (키워드 + 의미 검색)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, retriever],
    weights=[0.4, 0.6],
)

print("✅ 하이브리드 검색기 준비 완료")

✅ 하이브리드 검색기 준비 완료


### 4️⃣ 실험 — 고유명사/코드 검색 비교

**정책번호 "WM-RET-2024"** 처럼 의미가 없는 식별자는 벡터 검색이 약합니다.
`벡터` vs `BM25` vs `하이브리드` 를 비교해, 누가 정확히 찾아내는지 확인하세요.

> 🔧 실험: 질문을 `"1588-0000"`(전화번호), `"VIP 등급"`, `"신선식품 반품"` 등으로 바꿔 보세요.
> `weights=[0.4, 0.6]` 비율도 조절해 보세요.

In [ ]:
compare_search(
    "WM-RET-2024",
    {"벡터만": retriever, "BM25만": bm25, "하이브리드": ensemble_retriever},
)

❓ 질문: WM-RET-2024

── [벡터만] ────────────────────────────────
  📄 검색된 청크 3개
   [0] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024) ...
   [1] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)  [반품 신청 기간] 위니브마켓에서 ...
   [2] [환불 시점] 반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수 ...

── [BM25만] ────────────────────────────────
  📄 검색된 청크 3개
   [0] [회원 등급 혜택] 위니브마켓 회원은 실버, 골드, VIP 등급으로 나뉩니다. 등급이 높을수록 적립 ...
   [1] [배송 안내] 주문은 결제 완료 후 영업일 기준 1~2일 이내에 출고됩니다. 도서·산간 지역은 추가 ...
   [2] [환불 시점] 반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수 ...

── [하이브리드] ────────────────────────────────
  📄 검색된 청크 5개
   [0] [환불 시점] 반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수 ...
   [1] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024) ...
   [2] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)  [반품 신청 기간] 위니브마켓에서 ...
   [3] [회원 등급 혜택] 위니브마켓 회원은 실버, 골드, VIP 등급으로 나뉩니다. 등급이 높을수록 적립 ...
   [4] [배송 안내] 주문은 결제 완료 후 영업일 기준 1~2일 이내에 출고됩니다. 도서·산간 지역은 추가 ...



---
# 🎯 Part D. 실험 — MMR (다양성)  (5️⃣)

일반 검색은 **비슷한 청크만** 뽑기 쉽습니다. MMR은 **관련 있으면서 서로 다른** 청크를 뽑습니다.

### 5️⃣ 실험 — 일반 검색 vs MMR

`lambda_mult` : 1.0(관련성 우선) ↔ 0.0(다양성 우선). 기본 0.5.

> 🔧 실험: `lambda_mult`를 0.2 / 0.8로 바꿔 결과가 얼마나 다양해지는지 비교해 보세요.

In [ ]:
# MMR 리트리버
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5},
)

# 일반 vs MMR 비교 (넓은 질문일수록 차이가 잘 보임)
compare_search(
    "위니브마켓 정책 알려줘",
    {"일반 검색": retriever, "MMR": mmr_retriever},
)

❓ 질문: 위니브마켓 정책 알려줘

── [일반 검색] ────────────────────────────────
  📄 검색된 청크 3개
   [0] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024) ...
   [1] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)  [반품 신청 기간] 위니브마켓에서 ...
   [2] [반품 신청 기간] 위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습 ...

── [MMR] ────────────────────────────────
  📄 검색된 청크 3개
   [0] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024) ...
   [1] [반품 신청 기간] 위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습 ...
   [2] [회원 등급 혜택] 위니브마켓 회원은 실버, 골드, VIP 등급으로 나뉩니다. 등급이 높을수록 적립 ...



---
# 🔎 Part E. 실험 — 리랭킹 (심화·선택)  (6️⃣)

**넓게 검색(k=10) → LLM이 관련 없는 청크를 걸러냄.** 정확하지만 **느리고 비쌉니다.**

> ⚠️ 이 셀은 LLM 호출이 여러 번 일어나 **느릴 수 있습니다.** 선택 실험입니다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers.boolean import BooleanOutputParser
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainFilter

# 1단계: 넓게 검색 (k=10)
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 2단계: LLM이 관련 있는 청크만 통과시킴
filter_prompt = PromptTemplate(
    template=(
        "문서:\n{context}\n\n"
        "질문: {question}\n\n"
        "이 문서가 질문에 답하는 데 도움이 되면 YES, 아니면 NO. "
        "설명이나 다른 문구 없이 한 단어만 출력하세요."
    ),
    input_variables=["context", "question"],
    output_parser=BooleanOutputParser(),   # ← 반드시 함께
)

compressor = LLMChainFilter.from_llm(llm, prompt=filter_prompt)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=wide_retriever,
)

# 넓게(10개) vs 리랭킹 후(정선) 비교
q = "환불은 며칠 걸리나요?"
print("── [넓게 검색 k=10] " + "─"*25)
show_docs(wide_retriever.invoke(q))
print("\n── [리랭킹 후] " + "─"*30)
show_docs(compression_retriever.invoke(q))

── [넓게 검색 k=10] ─────────────────────────
  📄 검색된 청크 10개
   [0] [환불 시점] 반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행 ...
   [1] [반품 불가 상품] 신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다. 또한 고객의 사용으로 상품 가치가 ...
   [2] [반품 신청 기간] 위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다. ...
   [3] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024)  [반품 신청 기간] 위니브마켓에서 구매하신 상품은 수 ...
   [4] 위니브마켓에서 구매하신 상품은 수령일로부터 14일 이내에 반품을 신청할 수 있습니다. 반품 신청은 마이페이지 또는 고 ...
   [5] [배송 안내] 주문은 결제 완료 후 영업일 기준 1~2일 이내에 출고됩니다. 도서·산간 지역은 추가 배송비가 발생할 ...
   [6] [반품 배송비] 단순 변심에 의한 반품의 경우, 왕복 배송비는 고객님께서 부담하셔야 합니다. 다만 상품에 하자가 있거 ...
   [7] [반품 불가 상품] 신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다. 또한 고객의 사용으로 상품 가치가 ...
   [8] [교환 안내] 교환은 동일 상품의 색상 또는 사이즈 변경에 한해 1회 무료로 제공됩니다. 다른 상품으로의 교환은 반품 ...
   [9] 위니브마켓 고객 정책 안내 (정책번호: WM-RET-2024) ...

── [리랭킹 후] ──────────────────────────────
  📄 검색된 청크 2개
   [0] [환불 시점] 반품 상품이 물류센터에 도착해 검수가 완료되면, 영업일 기준 3일 이내에 결제하신 수단으로 환불이 진행 ...
   [1] [반품 불가 상품] 신선식품 및 냉장·냉동 식품은 상품 특성상 반품이 불가합니다. 또한 고객의 사용으로 상품 가치가 ...


---
# 🏷️ Part F. 실험 — 메타데이터 필터  (7️⃣)

청크에 **카테고리 꼬리표**를 달면, 검색 범위를 좁힐 수 있습니다.

### 7️⃣-a. 카테고리 메타데이터를 붙여 다시 저장

In [ ]:
from langchain_core.documents import Document

# 섹션별로 카테고리를 부여한 문서 만들기
sections = [
    ("반품", "반품은 수령일로부터 14일 이내 신청 가능하며, 고객센터(1588-0000)로 접수합니다."),
    ("반품", "단순 변심 반품은 왕복 배송비를 고객이 부담합니다. 하자·오배송은 위니브마켓이 부담합니다."),
    ("반품", "신선식품·냉장·냉동 식품은 반품이 불가합니다."),
    ("교환", "교환은 동일 상품의 색상·사이즈 변경에 한해 1회 무료로 제공됩니다."),
    ("환불", "반품 상품 검수 완료 후 영업일 기준 3일 이내 환불이 진행됩니다."),
    ("배송", "주문은 결제 완료 후 1~2일 이내 출고되며, 도서·산간은 추가 배송비가 있습니다."),
    ("회원", "회원은 실버·골드·VIP 등급으로 나뉘며, 등급이 높을수록 적립률·무료배송 혜택이 큽니다."),
]
docs_meta = [Document(page_content=text, metadata={"category": cat, "source": "정책서"})
             for cat, text in sections]

vectorstore_meta = make_store(documents=docs_meta, name="meta")
print(f"✅ 메타데이터 포함 저장 완료 — {len(docs_meta)}개 문서")

✅ 메타데이터 포함 저장 완료 — 7개 문서


### 7️⃣-b. 실험 — 필터 없음 vs 카테고리 필터

같은 질문이라도, 특정 카테고리 안에서만 검색하도록 좁힐 수 있습니다.

> 🔧 실험: `filter`의 카테고리를 `"환불"`, `"교환"` 등으로 바꿔 보세요.

In [ ]:
retriever_all = vectorstore_meta.as_retriever(search_kwargs={"k": 3})
retriever_filtered = vectorstore_meta.as_retriever(
    search_kwargs={"k": 3, "filter": {"category": "배송"}}
)

compare_search(
    "배송비는 얼마인가요?",
    {"필터 없음": retriever_all, "배송 카테고리만": retriever_filtered},
)

❓ 질문: 배송비는 얼마인가요?

── [필터 없음] ────────────────────────────────
  📄 검색된 청크 3개
   [0] 주문은 결제 완료 후 1~2일 이내 출고되며, 도서·산간은 추가 배송비가 있습니다. ...   {'source': '정책서', 'category': '배송'}
   [1] 단순 변심 반품은 왕복 배송비를 고객이 부담합니다. 하자·오배송은 위니브마켓이 부담합니다. ...   {'source': '정책서', 'category': '반품'}
   [2] 회원은 실버·골드·VIP 등급으로 나뉘며, 등급이 높을수록 적립률·무료배송 혜택이 큽니다. ...   {'category': '회원', 'source': '정책서'}

── [배송 카테고리만] ────────────────────────────────
  📄 검색된 청크 1개
   [0] 주문은 결제 완료 후 1~2일 이내 출고되며, 도서·산간은 추가 배송비가 있습니다. ...   {'category': '배송', 'source': '정책서'}



### 7️⃣-c. (보너스) 유사도 임계값 — 관련 없으면 안 가져오기

점수가 낮은(관련 없는) 청크는 아예 반환하지 않게 할 수 있습니다.
문서에 **없는 질문**을 넣으면 빈 결과가 나오는지 확인해 보세요.
(임계값은 문서·모델마다 다르니 값을 조절해야 합니다.)

In [ ]:
retriever_thresh = vectorstore_meta.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.4, "k": 3},
)

print("── 문서에 있는 질문 ──")
show_docs(retriever_thresh.invoke("반품 기간"))
print("\n── 문서에 없는 질문 (해외 배송) ──")
show_docs(retriever_thresh.invoke("해외 배송 비행기 요금"))

── 문서에 있는 질문 ──
  📄 검색된 청크 3개
   [0] 반품은 수령일로부터 14일 이내 신청 가능하며, 고객센터(1588-0000)로 접수합니다. ...   {'source': '정책서', 'category': '반품'}
   [1] 반품 상품 검수 완료 후 영업일 기준 3일 이내 환불이 진행됩니다. ...   {'category': '환불', 'source': '정책서'}
   [2] 단순 변심 반품은 왕복 배송비를 고객이 부담합니다. 하자·오배송은 위니브마켓이 부담합니다. ...   {'category': '반품', 'source': '정책서'}

── 문서에 없는 질문 (해외 배송) ──
  📄 검색된 청크 3개
   [0] 주문은 결제 완료 후 1~2일 이내 출고되며, 도서·산간은 추가 배송비가 있습니다. ...   {'source': '정책서', 'category': '배송'}
   [1] 단순 변심 반품은 왕복 배송비를 고객이 부담합니다. 하자·오배송은 위니브마켓이 부담합니다. ...   {'source': '정책서', 'category': '반품'}
   [2] 신선식품·냉장·냉동 식품은 반품이 불가합니다. ...   {'category': '반품', 'source': '정책서'}


---
# 📝 Part G. 실험 — 프롬프트 고도화  (8️⃣)

검색을 잘해도 프롬프트가 약하면 LLM은 지어냅니다. **추측 금지 + 예시(few-shot)** 로 환각을 막습니다.

### 8️⃣ 실험 — 기본 프롬프트 vs 개선 프롬프트

**문서에 없는 질문**("해외 배송")을 던져, 두 프롬프트의 답이 어떻게 다른지 비교하세요.
개선 프롬프트는 "안내되어 있지 않습니다"라고 정직하게 답해야 합니다.

> 🔧 실험: 질문을 문서에 있는 것/없는 것으로 바꿔가며 두 답을 비교해 보세요.

In [ ]:
# 개선 프롬프트: 역할 + 추측 금지 + few-shot 예시
improved_prompt = ChatPromptTemplate.from_template(
    "당신은 위니브마켓 고객센터 상담원입니다.\n"
    "규칙:\n"
    "1) 반드시 아래 [컨텍스트]에 있는 내용만으로 답하세요.\n"
    "2) 컨텍스트에 없으면 추측하지 말고 '해당 내용은 안내되어 있지 않습니다'라고 답하세요.\n"
    "3) 답변은 3문장 이내로 간결하게.\n\n"
    "[예시]\n"
    "질문: 적립금은 어떻게 쌓이나요?\n"
    "답변: 죄송하지만 제공된 문서에는 적립금 적립 방법이 안내되어 있지 않습니다.\n\n"
    "[컨텍스트]\n{context}\n\n[질문]\n{question}"
)

# 기본 체인 vs 개선 체인
basic_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
               | basic_prompt | llm | StrOutputParser())
improved_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
                  | improved_prompt | llm | StrOutputParser())

question = "해외 배송은 얼마인가요?"   # 문서에 없는 질문
print("── [기본 프롬프트] ──")
ask(question, basic_chain)
print("── [개선 프롬프트] ──")
ask(question, improved_chain)

── [기본 프롬프트] ──
❓ 해외 배송은 얼마인가요?
🤖 모릅니다.

── [개선 프롬프트] ──
❓ 해외 배송은 얼마인가요?
🤖 죄송하지만 제공된 문서에는 해외 배송 비용에 관한 내용이 안내되어 있지 않습니다.



---
# 📎 Part H. 실험 — 출처 표시  (9️⃣)

답변과 함께 **어느 문서에서 나왔는지(출처)** 를 반환합니다. `RunnableParallel`을 사용합니다.
(F에서 만든 메타데이터 저장소를 써서 카테고리·출처가 보이게 합니다.)

### 9️⃣ 실험 — 답변 + 출처 함께 받기

In [ ]:
from langchain_core.runnables import RunnableParallel

retriever_meta = vectorstore_meta.as_retriever(search_kwargs={"k": 2})

# 답변 생성 부분 (context 문서 → 문자열 → 프롬프트 → LLM)
answer_chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | improved_prompt | llm | StrOutputParser()
)

# 검색 결과(출처) + 답변을 함께 반환
rag_with_source = RunnableParallel(
    context=retriever_meta,
    question=RunnablePassthrough(),
).assign(answer=answer_chain)

# 실행
result = rag_with_source.invoke("반품은 며칠 이내에 가능한가요?")
print("🤖 답변:", result["answer"])
print("\n📎 출처:")
for d in result["context"]:
    print("  -", d.metadata, "|", d.page_content[:30], "...")

🤖 답변: 반품은 상품 수령일로부터 14일 이내에 신청 가능합니다. 고객센터(1588-0000)를 통해 접수해 주시기 바랍니다.

📎 출처:
  - {'source': '정책서', 'category': '반품'} | 반품은 수령일로부터 14일 이내 신청 가능하며, 고객센 ...
  - {'source': '정책서', 'category': '환불'} | 반품 상품 검수 완료 후 영업일 기준 3일 이내 환불이 ...


---
# 🧪 자유 실험 놀이터

아래 아이디어를 직접 바꿔가며 실험해 보세요.

1. **청킹** — B에서 `chunk_size`를 50 / 500 / 1000 으로 바꾸면 검색이 어떻게 달라지나요?
2. **하이브리드 가중치** — C에서 `weights=[0.7, 0.3]`(키워드 우선)로 바꾸면 코드 검색이 더 잘 되나요?
3. **MMR** — D에서 `lambda_mult`를 0.1 vs 0.9로 바꿔 다양성 차이를 보세요.
4. **필터** — F에서 여러 카테고리(`반품`/`환불`/`배송`)로 필터링해 보세요.
5. **프롬프트** — G의 개선 프롬프트에 "답변 끝에 관련 카테고리를 표시하라" 같은 규칙을 추가해 보세요.
6. **나만의 문서** — A-1의 `policy_text`를 여러분의 문서로 바꿔, 전체 실험을 다시 돌려보세요.

> 💡 **원칙**: 한 번에 하나씩 바꾸고, `compare_search`/`ask`로 **적용 전후를 비교**하세요.

In [ ]:
# ✏️ 여기서 자유롭게 실험!
# 예시)
# compare_search("VIP 등급 혜택", {"벡터": retriever, "하이브리드": ensemble_retriever})
# ask("교환은 어떻게 하나요?", advanced_chain)
